# Inspect a teammate's `density_auto` predictions

This is an imitation diagnostic, **not a Task 5 competition model**. It uses the supplied prediction CSV as a target to find a compact explanation of its output. Do not interpret agreement with those predictions as anomaly accuracy or F3. No model artifact or submission is written.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pipeline/save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
SRC = ROOT / "src"
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from src.pipeline.random_seed import set_seed
SEED = set_seed()
FRIEND = Path.home() / "Downloads" / "kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density_auto_v1.csv"
friend = pd.read_csv(FRIEND)
card = json.loads((SRC / "task5/data/eda/eda_original_v2.json").read_text(encoding="utf-8"))
X = pd.read_parquet(SRC / card["splits"]["test"]["path"]).drop(columns="player_id")
sample = pd.read_csv(SRC / "submission/sample_submission.csv", usecols=["id"])
assert len(X) == len(friend) == len(sample) == 25889 and friend.id.equals(sample.id)
y = friend.task5.to_numpy(dtype=int)
assert set(y) == {0,1}
print("friend alerts",int(y.sum()),"of",len(y))

friend alerts 2046 of 25889


In [2]:
train_idx, valid_idx = train_test_split(np.arange(len(y)), test_size=0.3, stratify=y, random_state=SEED)
imputer = SimpleImputer(strategy="median").fit(X.iloc[train_idx])
X_train = imputer.transform(X.iloc[train_idx]); X_valid = imputer.transform(X.iloc[valid_idx])
surrogate = ExtraTreesClassifier(n_estimators=160, min_samples_leaf=3, max_features=0.7,
                                 n_jobs=-1, random_state=SEED).fit(X_train, y[train_idx])
prob = surrogate.predict_proba(X_valid)[:,1]
k = int(y[valid_idx].sum())
chosen = np.argsort(-prob,kind="stable")[:k]
print("held-out rows",len(valid_idx),"teammate alerts",k,"same-budget matching alerts",int(y[valid_idx][chosen].sum()))
print("ROC AUC",roc_auc_score(y[valid_idx],prob),"average precision",average_precision_score(y[valid_idx],prob))
importance = pd.Series(surrogate.feature_importances_,index=X.columns).sort_values(ascending=False)
display(importance.head(12).to_frame("importance"))

held-out rows 7767 teammate alerts 614 same-budget matching alerts 597
ROC AUC 0.9954408550932594 average precision 0.9911105720684877


,importance
location_changes_4,0.157799
behavior_stability,0.133426
friend_removes_4,0.079677
skill_trajectory_4,0.059178
activity_level_3,0.052658
currency_transfers_4,0.038757
communication_frequency_4,0.036810
communication_frequency_1,0.035673
activity_level_4,0.033925
activity_level_2,0.030230


In [3]:
rule_tree = DecisionTreeClassifier(max_depth=7,min_samples_leaf=20,random_state=SEED).fit(X_train,y[train_idx])
rule_pred = rule_tree.predict(X_valid)
print("Small-tree held-out F1 against teammate predictions",f1_score(y[valid_idx],rule_pred))
print(export_text(rule_tree,feature_names=list(X.columns),max_depth=7))

Small-tree held-out F1 against teammate predictions 0.9342560553633218
|--- location_changes_4 <= 4.31
|   |--- skill_trajectory_4 <= 36.25
|   |   |--- behavior_stability <= 0.55
|   |   |   |--- class: 1
|   |   |--- behavior_stability >  0.55
|   |   |   |--- activity_level_3 <= 0.41
|   |   |   |   |--- class: 1
|   |   |   |--- activity_level_3 >  0.41
|   |   |   |   |--- communication_frequency_1 <= 0.17
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- communication_frequency_1 >  0.17
|   |   |   |   |   |--- session_duration_std_4 <= 6.76
|   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |--- session_duration_std_4 >  6.76
|   |   |   |   |   |   |--- accuracy_std_4 <= 11.38
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |--- accuracy_std_4 >  11.38
|   |   |   |   |   |   |   |--- class: 1
|   |--- skill_trajectory_4 >  36.25
|   |   |--- class: 1
|--- location_changes_4 >  4.31
|   |--- purchase_amount_mean_3 <= 493.49
|   |   |--- ip_hash_en

The surrogate is fitted to a teammate's predictions, so it is **not** an independently trained unsupervised detector and must not be treated as evidence of Task 5 F3. Its purpose is to test whether the prediction pattern is simple enough to describe while waiting for the actual `density_auto` code or model card.